# Bibliotecas

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.colors as pc
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifierCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, normalized_mutual_info_score, make_scorer, f1_score, precision_score, recall_score
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_validate
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors

from scipy.signal import find_peaks
from scipy.integrate import trapezoid
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import skew, kurtosis, gaussian_kde, mannwhitneyu, ks_2samp, pearsonr, spearmanr, kendalltau
from scipy import stats

from hampel import hampel

from aeon.transformations.collection.convolution_based import Rocket

from tslearn.clustering import KShape
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.utils import to_time_series_dataset

warnings.filterwarnings('ignore')

DIR_DATA = os.getcwd()+"/data/"
DIR_OUTPUT = os.getcwd()+"/output/"

# Carregando dados

## Crystallizer #1

In [ ]:
base_name_crystallizer1 = "Crystallizer #1.csv"

df_crystallizer1 = pd.read_csv(DIR_DATA + base_name_crystallizer1, sep=";", decimal=".")
df_crystallizer1["TIMESTAMP"] = pd.to_datetime(df_crystallizer1["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer1["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer1["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer1.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer1 = df_crystallizer1[df_crystallizer1.duplicated(subset=['Labref'], keep=False)]

# Retirando linhas 23 e 2141 que estão duplicadas mas não possuem amostras significativas
df_crystallizer1.drop([23,2141], inplace=True) 
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer1[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer1.columns if col != 'Labref'}
df_crystallizer1 = df_crystallizer1.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer1 = df_crystallizer1.drop(remove.index)
df_crystallizer1

### Removendo outliers 0 a 10 #1

In [ ]:
df_crystallizer1_0a10 = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer1)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer1_0a10)} ({len(df_crystallizer1) - len(df_crystallizer1_0a10)} removidas)")

df_crystallizer1_0a10

### Removendo outliers com IQR #1

In [ ]:
serie = df_crystallizer1["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer1_iqr = df_crystallizer1[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer1)}")
print(f"Amostras removidas: {len(df_crystallizer1) - len(df_crystallizer1_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer1_iqr)}")

df_crystallizer1_iqr

### Removendo outliers com Filtro de Hampel #1

In [ ]:
serie = df_crystallizer1["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer1_hampel = df_crystallizer1.copy().reset_index(drop=True)
df_crystallizer1_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer1_hampel

## Crystallizer #2

In [ ]:
base_name_crystallizer2 = "Crystallizer #2.csv"

df_crystallizer2 = pd.read_csv(DIR_DATA + base_name_crystallizer2, sep=";", decimal=".")
df_crystallizer2["TIMESTAMP"] = pd.to_datetime(df_crystallizer2["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer2["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer2["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer2.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer2 = df_crystallizer2[df_crystallizer2.duplicated(subset=['Labref'], keep=False)]

# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer2[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer2.columns if col != 'Labref'}
df_crystallizer2 = df_crystallizer2.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostra 4027521 que possui valor muito discrepante
idx = df_crystallizer2[df_crystallizer2['Labref'] == 4027521].index 
df_crystallizer2 = df_crystallizer2.drop(idx)


# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer2 = df_crystallizer2.drop(remove.index)
df_crystallizer2

### Removendo outliers 0 a 10 #2

In [ ]:
df_crystallizer2_0a10 = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer2)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer2_0a10)} ({len(df_crystallizer2) - len(df_crystallizer2_0a10)} removidas)")

df_crystallizer2_0a10

### Removendo outliers com IQR #2

In [ ]:
serie = df_crystallizer2["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer2_iqr = df_crystallizer2[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer2)}")
print(f"Amostras removidas: {len(df_crystallizer2) - len(df_crystallizer2_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer2_iqr)}")

df_crystallizer2_iqr

### Removendo outliers com Filtro de Hampel #2

In [ ]:
serie = df_crystallizer2["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer2_hampel = df_crystallizer2.copy().reset_index(drop=True)
df_crystallizer2_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer2_hampel

## Crystallizer #3

In [ ]:
base_name_crystallizer3 = "Crystallizer #3.csv"

df_crystallizer3 = pd.read_csv(DIR_DATA + base_name_crystallizer3, sep=";", decimal=".")
df_crystallizer3["TIMESTAMP"] = pd.to_datetime(df_crystallizer3["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer3["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer3["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer3.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer3 = df_crystallizer3[df_crystallizer3.duplicated(subset=['Labref'], keep=False)]


# Retirando linhas 6476 e 6494 que estão duplicadas mas não possuem amostras significativas
df_crystallizer3.drop([6476,6494], inplace=True) 
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer3[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer3.columns if col != 'Labref'}
df_crystallizer3 = df_crystallizer3.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer3 = df_crystallizer3.drop(remove.index)
df_crystallizer3

### Removendo outliers 0 a 10 #3

In [ ]:
df_crystallizer3_0a10 = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"].between(0, 10)].reset_index(drop=True)

print(f"Amostras originais : {len(df_crystallizer3)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer3_0a10)} ({len(df_crystallizer3) - len(df_crystallizer3_0a10)} removidas)")

df_crystallizer3_0a10

### Removendo outliers com IQR #3

In [ ]:
serie = df_crystallizer3["Resultado de Ferro (ppm)"]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

df_crystallizer3_iqr = df_crystallizer3[serie.between(limite_inf, limite_sup)].reset_index(drop=True)

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer3)}")
print(f"Amostras removidas: {len(df_crystallizer3) - len(df_crystallizer3_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer3_iqr)}")

df_crystallizer3_iqr

### Removendo outliers com Filtro de Hampel #3

In [ ]:
serie = df_crystallizer3["Resultado de Ferro (ppm)"].reset_index(drop=True)

resultado = hampel(serie, window_size=7, n_sigma=3.0)

serie_filtrada = resultado.filtered_data
outlier_indices = resultado.outlier_indices

df_crystallizer3_hampel = df_crystallizer3.copy().reset_index(drop=True)
df_crystallizer3_hampel["Resultado de Ferro (ppm)"] = serie_filtrada

print(f"Outliers detectados: {len(outlier_indices)}")
df_crystallizer3_hampel

# Carregando eventos identificados

## Deifinindo LC

In [ ]:
threshold = 5

## Crystallizer #1

In [ ]:
base_name_eventos_crystallizer1 = "Eventos-Reator1.csv"

df_eventos_crystallizer1 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer1, sep=";", decimal=".")
df_eventos_crystallizer1["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer1["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer1["Real"] = 1

nova_linha = {
    "TIMESTAMP": pd.to_datetime("2025-10-21 08:38:00"),
    "Real": 0,
    "Evento": "Falso Alarme"
}

df_eventos_crystallizer1 = pd.concat([df_eventos_crystallizer1, pd.DataFrame([nova_linha])], ignore_index=True)
df_eventos_crystallizer1 = df_eventos_crystallizer1.sort_values("TIMESTAMP").reset_index(drop=True)

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer1.drop([2,3,4,6,7,8], inplace=True) 
df_eventos_crystallizer1

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer1 original
    diffs = (df_eventos_crystallizer1["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer1 existente
df_eventos_crystallizer1 = pd.concat([df_eventos_crystallizer1, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer1 = df_eventos_crystallizer1.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer1

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer1_filtrado_until2020 = df_eventos_crystallizer1[df_eventos_crystallizer1["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer1_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer1_filtrado_after2020 = df_eventos_crystallizer1[df_eventos_crystallizer1["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer1_filtrado_after2020.head()

## Crystallizer #2

In [ ]:
base_name_eventos_crystallizer2 = "Eventos-Reator2.csv"

df_eventos_crystallizer2 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer2, sep=";", decimal=".")
df_eventos_crystallizer2["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer2["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer2["Real"] = 1

# Removendo eventos fora do período de dados
df_eventos_crystallizer2.drop([0,1,2], inplace=True) 

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer2.drop([6,9], inplace=True) 
df_eventos_crystallizer2

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer2 original
    diffs = (df_eventos_crystallizer2["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer2 existente
df_eventos_crystallizer2 = pd.concat([df_eventos_crystallizer2, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer2 = df_eventos_crystallizer2.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer2

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer2_filtrado_until2020 = df_eventos_crystallizer2[df_eventos_crystallizer2["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer2_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer2_filtrado_after2020 = df_eventos_crystallizer2[df_eventos_crystallizer2["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer2_filtrado_after2020.head()

## Crystallizer #3

In [ ]:
base_name_eventos_crystallizer3 = "Eventos-Reator3.csv"

df_eventos_crystallizer3 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer3, sep=";", decimal=".")
df_eventos_crystallizer3["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer3["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer3["Real"] = 1

df_eventos_crystallizer3.drop([0,1], inplace=True) # Removendo eventos fora do período de dados

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer3.drop([4,7], inplace=True) 
df_eventos_crystallizer3

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer3 original
    diffs = (df_eventos_crystallizer3["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer3 existente
df_eventos_crystallizer3 = pd.concat([df_eventos_crystallizer3, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer3 = df_eventos_crystallizer3.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer3

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer3_filtrado_until2020 = df_eventos_crystallizer3[df_eventos_crystallizer3["TIMESTAMP"] <= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer3_filtrado_until2020.head()

In [ ]:
df_eventos_crystallizer3_filtrado_after2020 = df_eventos_crystallizer3[df_eventos_crystallizer3["TIMESTAMP"] >= pd.to_datetime("2020-04-01")].reset_index(drop=True)
df_eventos_crystallizer3_filtrado_after2020.head()

# Plotando Gráfico das medições
Linhas vermelhas = Eventos relatados  
Linhas azuis = Eventos de ultapassagem sem relatos

## Crystallizer #1

In [ ]:
fig_crystallizer1 = go.Figure()
fig_crystallizer1.add_trace(go.Scatter(
    x=df_crystallizer1['TIMESTAMP'],
    y=df_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1",
)
fig_crystallizer1.show()

### Crystallizer #1 - Outliers 0 a 10

In [ ]:
fig_crystallizer1_0a10 = go.Figure()
fig_crystallizer1_0a10.add_trace(go.Scatter(
    x=df_crystallizer1_0a10['TIMESTAMP'],
    y=df_crystallizer1_0a10["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1_0a10.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1_0a10.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1_0a10.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1_0a10.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1 - Outliers 0 a 10",
)
fig_crystallizer1_0a10.show()

### Crystallizer #1 - Outliers IQR

In [ ]:
fig_crystallizer1_iqr = go.Figure()
fig_crystallizer1_iqr.add_trace(go.Scatter(
    x=df_crystallizer1_iqr['TIMESTAMP'],
    y=df_crystallizer1_iqr["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1_iqr.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1_iqr.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1_iqr.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1_iqr.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1 - IQR",
)
fig_crystallizer1_iqr.show()

### Crystallizer #1 - Filtro de Hampel

In [ ]:
fig_crystallizer1_hampel = go.Figure()
fig_crystallizer1_hampel.add_trace(go.Scatter(
    x=df_crystallizer1_hampel['TIMESTAMP'],
    y=df_crystallizer1_hampel["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1_hampel.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1_hampel.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1_hampel.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1_hampel.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1 - Filtro de Hampel",
)
fig_crystallizer1_hampel.show()

## Crystallizer #2

In [ ]:
fig_crystallizer2 = go.Figure()
fig_crystallizer2.add_trace(go.Scatter(
    x=df_crystallizer2['TIMESTAMP'],
    y=df_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2",
)
fig_crystallizer2.show()

### Crystallizer #2 - Outliers 0 a 10

In [ ]:
fig_crystallizer2_0a10 = go.Figure()
fig_crystallizer2_0a10.add_trace(go.Scatter(
    x=df_crystallizer2_0a10['TIMESTAMP'],
    y=df_crystallizer2_0a10["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2_0a10.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2_0a10.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2_0a10.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2_0a10.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2 - Outliers 0 a 10",
)
fig_crystallizer2_0a10.show()

### Crystallizer #2 - Outliers IQR

In [ ]:
fig_crystallizer2_iqr = go.Figure()
fig_crystallizer2_iqr.add_trace(go.Scatter(
    x=df_crystallizer2_iqr['TIMESTAMP'],
    y=df_crystallizer2_iqr["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2_iqr.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2_iqr.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2_iqr.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2_iqr.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2 - IQR",
)
fig_crystallizer2_iqr.show()

### Crystallizer #2 - Filtro de Hampel

In [ ]:
fig_crystallizer2_hampel = go.Figure()
fig_crystallizer2_hampel.add_trace(go.Scatter(
    x=df_crystallizer2_hampel['TIMESTAMP'],
    y=df_crystallizer2_hampel["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2_hampel.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2_hampel.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2_hampel.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2_hampel.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2 - Filtro de Hampel",
)
fig_crystallizer2_hampel.show()

## Crystallizer #3

In [ ]:
fig_crystallizer3 = go.Figure()
fig_crystallizer3.add_trace(go.Scatter(
    x=df_crystallizer3['TIMESTAMP'],
    y=df_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3",
)
fig_crystallizer3.show()

### Crystallizer #3 - Outliers 0 a 10

In [ ]:
fig_crystallizer3_0a10 = go.Figure()
fig_crystallizer3_0a10.add_trace(go.Scatter(
    x=df_crystallizer3_0a10['TIMESTAMP'],
    y=df_crystallizer3_0a10["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3_0a10.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3_0a10.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3_0a10.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3_0a10.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3 - Outliers 0 a 10",
)
fig_crystallizer3_0a10.show()

### Crystallizer #3 - Outliers IQR

In [ ]:
fig_crystallizer3_iqr = go.Figure()
fig_crystallizer3_iqr.add_trace(go.Scatter(
    x=df_crystallizer3_iqr['TIMESTAMP'],
    y=df_crystallizer3_iqr["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3_iqr.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3_iqr.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3_iqr.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3_iqr.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3 - IQR",
)
fig_crystallizer3_iqr.show()

### Crystallizer #2 - Filtro de Hampel

In [ ]:
fig_crystallizer3_hampel = go.Figure()
fig_crystallizer3_hampel.add_trace(go.Scatter(
    x=df_crystallizer3_hampel['TIMESTAMP'],
    y=df_crystallizer3_hampel["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3_hampel.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3_hampel.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3_hampel.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3_hampel.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3 - Filtro de Hampel",
)
fig_crystallizer3_hampel.show()

## Crystallizer #1#2#3

In [ ]:
crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3"
)
fig.show()

### Outliers 0 a 10

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_0a10, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_0a10, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_0a10, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3 - Outliers 0 a 10"
)
fig.show()

### Outliers IQR

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_iqr, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_iqr, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_iqr, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3 - Outliers IQR"
)
fig.show()

### Outliers Hampel

In [ ]:
crystallizers = [
    {"df": df_crystallizer1_hampel, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2_hampel, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3_hampel, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3 - Filtro de Hampel"
)
fig.show()

## Gráfico com Média Móvel
Verificando se há tendência clara nos dados

## MM Crystallizer #1

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1"
)
fig_mm_crystallizer1.show()

### MM Crystallizer #1 - Outliers 0 a 10

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1_0a10.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1 - Outliers 0 a 10"
)
fig_mm_crystallizer1.show()

### MM Crystallizer #1 - IQR

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1_iqr.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1 - IQR"
)
fig_mm_crystallizer1.show()

### MM Crystallizer #1 - Filtro de Hampel

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1_hampel.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1 - Filtro de Hampel"
)
fig_mm_crystallizer1.show()

## MM Crystallizer #2

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2"
)
fig_mm_crystallizer2.show()

### MM Crystallizer #2 - Outliers 0 a 10

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2_0a10.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2 - Outiliers 0 a 10"
)
fig_mm_crystallizer2.show()

### MM Crystallizer #2 - IQR

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2_iqr.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2 - IQR"
)
fig_mm_crystallizer2.show()

### MM Crystallizer #2 - Filtro de Hampel

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2_hampel.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2 - Filtro de Hampel"
)
fig_mm_crystallizer2.show()

## MM Crystallizer #3

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3"
)
fig_mm_crystallizer3.show()

### MM Crystallizer #3 - Outliers 0 a 10

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3_0a10.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3 - Outliers 0 a 10"
)
fig_mm_crystallizer3.show()

### MM Crystallizer #3 - IQR

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3_iqr.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3 - IQR"
)
fig_mm_crystallizer3.show()

### MM Crystallizer #3 - Filtro de Hampel

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3_hampel.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3 - Filtro de Hampel"
)
fig_mm_crystallizer3.show()

## MM Crystallizer #1#2#3

In [ ]:
NUM_DIAS = 7  # parâmetro da janela

CORES = {
    "Crystallizer #1": "blue",
    "Crystallizer #2": "green",
    "Crystallizer #3": "orange"
}

crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3"},
]

fig_mm_123 = go.Figure()

for c in crystallizers:
    cor = CORES[c["nome"]]

    # Calcula média móvel
    df_mm = c["df"].copy().set_index('TIMESTAMP')
    df_mm[f'MM_{NUM_DIAS}D'] = df_mm["Resultado de Ferro (ppm)"].rolling(window=f'{NUM_DIAS}D').mean()
    df_mm = df_mm.reset_index()

    # Série original
    fig_mm_123.add_trace(go.Scatter(
        x=df_mm['TIMESTAMP'],
        y=df_mm["Resultado de Ferro (ppm)"],
        mode='lines',
        name=f"{c['nome']} — original",
        line=dict(color=cor, width=1),
        opacity=0.3
    ))

    # Média móvel
    fig_mm_123.add_trace(go.Scatter(
        x=df_mm['TIMESTAMP'],
        y=df_mm[f'MM_{NUM_DIAS}D'],
        mode='lines',
        name=f"{c['nome']} — MM {NUM_DIAS}D",
        line=dict(color=cor, width=2)
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        dash_evento = "solid" if row["Real"] == 1 else "dash"
        fig_mm_123.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0, y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash=dash_evento)
        )
        fig_mm_123.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig_mm_123.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig_mm_123.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title=f"Fe (ppm) MM {NUM_DIAS}D — Crystallizers #1, #2 e #3"
)
fig_mm_123.show()

# Estatísticas descritivas de toda série de concentração de Fe

In [ ]:
def estatisticas(serie, nome):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    return pd.Series({
        "Contagem"     : serie.count(),
        "Média"        : serie.mean(),
        "Mediana"      : serie.median(),
        "Desvio Padrão": serie.std(),
        "Variância"    : serie.var(),
        "Mínimo"       : serie.min(),
        "Máximo"       : serie.max(),
        "Amplitude"    : serie.max() - serie.min(),
        "Q1 (25%)"     : Q1,
        "Q3 (75%)"     : Q3,
        "IQR"          : Q3 - Q1,
        "Assimetria"   : serie.skew(),
        "Curtose"      : serie.kurt()
    }, name=nome)

# Coluna separadora vazia
separador = pd.Series({k: "" for k in ["Contagem","Média","Mediana","Desvio Padrão","Variância",
                                        "Mínimo","Máximo","Amplitude","Q1 (25%)","Q3 (75%)","IQR",
                                        "Assimetria","Curtose"]})

c1 = pd.concat([
    estatisticas(df_crystallizer1["Resultado de Ferro (ppm)"].dropna(),       "C1 Original"),
    estatisticas(df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna(),  "C1 Intervalo 0-10"),
    estatisticas(df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna(),   "C1 IQR"),
    estatisticas(df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna(),"C1 Hampel"),
], axis=1)

c2 = pd.concat([
    estatisticas(df_crystallizer2["Resultado de Ferro (ppm)"].dropna(),       "C2 Original"),
    estatisticas(df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna(),  "C2 Intervalo 0-10"),
    estatisticas(df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna(),   "C2 IQR"),
    estatisticas(df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna(),"C2 Hampel"),
], axis=1)

c3 = pd.concat([
    estatisticas(df_crystallizer3["Resultado de Ferro (ppm)"].dropna(),       "C3 Original"),
    estatisticas(df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna(),  "C3 Intervalo 0-10"),
    estatisticas(df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna(),   "C3 IQR"),
    estatisticas(df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna(),"C3 Hampel"),
], axis=1)

sep = separador.rename("│")

df_comparativo = pd.concat([c1, sep, c2, sep.rename("│"), c3], axis=1).round(4)

# Corrige as colunas separadoras que ficaram com float após o round
df_comparativo["│"]  = ""
df_comparativo["│"] = ""

df_comparativo

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

CORES = {"C1": "#1f77b4", "C2": "#2ca02c", "C3": "#ff7f0e"}

def hex_to_rgba(cor, alpha=0.15):
    rgb = pc.hex_to_rgb(cor)
    return f'rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha})'

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=2,
    subplot_titles=[
        titulo
        for m in metodos
        for titulo in [f"KDE — {m['titulo']}", f"Violin Plot — {m['titulo']}"]
    ]
)

for row_idx, metodo in enumerate(metodos, start=1):
    for c in metodo["series"]:
        serie = c["serie"]
        nome  = c["nome"]
        cor   = CORES[nome]

        # KDE na escala de densidade natural (área sob a curva = 1)
        kde = gaussian_kde(serie)
        x_range = np.linspace(serie.min(), serie.max(), 1000)
        y_kde = kde(x_range)
        y_kde = y_kde / trapezoid(y_kde, x_range)  # normaliza

        fig.add_trace(go.Scatter(
            x=x_range,
            y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=hex_to_rgba(cor, alpha=0.15),
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Violin
        fig.add_trace(go.Violin(
            y=serie,
            name=nome,
            marker_color=cor,
            fillcolor=hex_to_rgba(cor, alpha=0.4),
            box_visible=True,
            meanline_visible=True,
            legendgroup=nome,
            showlegend=False
        ), row=row_idx, col=2)

    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)
    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_yaxes(title_text="ppm", row=row_idx, col=2)

fig.update_layout(
    height=500 * n_metodos,
    template='plotly_white',
    title="Análise Descritiva Comparativa — Resultado de Ferro (ppm)",
)
fig.show()

### Teste de normalidade Q-Q Plot

In [ ]:
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

serie = df["Resultado de Ferro (ppm)"].dropna()

# Visualização
fig_normalidade_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie.min(), serie.max(), 300)
y_normal = stats.norm.pdf(x_range, serie.mean(), serie.std())
y_normal_scaled = y_normal * len(serie) * (serie.max() - serie.min()) / 50

fig_normalidade_crystallizer1.add_trace(go.Histogram(
    x=serie, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer1.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer1.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer1.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer1.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer1.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #1"
)
fig_normalidade_crystallizer1.show()

# Estatísticas descritivas dos eventos

## Crystallizer #1

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer1_0a10},
    {"titulo": "IQR",              "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer1.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

registros = []

for DIAS_JANELA in JANELAS:
    stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}

    for _, evento in df_eventos_crystallizer1.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) < 2:
            continue
        classe = int(evento["Real"])
        for nome_stat, func in STATS_FUNCS.items():
            stat_vals[nome_stat][classe].append(func(v))

    for nome_stat in STATS_FUNCS:
        v0 = np.array(stat_vals[nome_stat][0])
        v1 = np.array(stat_vals[nome_stat][1])
        if len(v0) < 2 or len(v1) < 2:
            continue
        stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
        effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
        registros.append({
            'janela':      f"{DIAS_JANELA}d",
            'feature':     nome_stat,
            'effect_size': round(effect, 4),
            'p_value':     round(p, 4),
        })

df_effect = pd.DataFrame(registros)

# Heatmap de effect size
pivot = df_effect.pivot(index='feature', columns='janela', values='effect_size')
pivot = pivot[[f"{d}d" for d in JANELAS]]  # ordena colunas

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    text=np.round(pivot.values, 3),
    texttemplate="%{text}",
    colorbar=dict(title="Effect Size")
))
fig.update_layout(
    title="Effect Size (Mann-Whitney) por Feature e Janela<br>"
          "<sup>Verde = alta separabilidade entre Real=0 e Real=1</sup>",
    template="plotly_white",
    xaxis_title="Janela", yaxis_title="Feature"
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}
n_metodos = len(metodos)

fig = make_subplots(
    rows=n_metodos, cols=len(JANELAS),
    subplot_titles=[
        f"{m['titulo']} — {d} dias"
        for m in metodos
        for d in JANELAS
    ],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
        dados_por_classe = {0: [], 1: []}

        for _, evento in df_eventos_crystallizer1.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
            valores = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(valores) < 2:
                continue
            classe = int(evento["Real"])
            dados_por_classe[classe].extend(valores.tolist())

        for classe, valores in dados_por_classe.items():
            fig.add_trace(go.Box(
                y=valores,
                name=nomes_classe[classe],
                marker_color=cores_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and col_idx == 1),
                boxmean='sd'
            ), row=row_idx, col=col_idx)

        fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
        fig.update_xaxes(title_text=metodo["titulo"], row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    boxmode="group",
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela — Crystallizer #1"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer1_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 (Eventos após 01/04/2020)"
)
fig.show()

## Crystallizer #2

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer2_0a10},
    {"titulo": "IQR",              "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer2.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

registros = []

for DIAS_JANELA in JANELAS:
    stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}

    for _, evento in df_eventos_crystallizer2.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) < 2:
            continue
        classe = int(evento["Real"])
        for nome_stat, func in STATS_FUNCS.items():
            stat_vals[nome_stat][classe].append(func(v))

    for nome_stat in STATS_FUNCS:
        v0 = np.array(stat_vals[nome_stat][0])
        v1 = np.array(stat_vals[nome_stat][1])
        if len(v0) < 2 or len(v1) < 2:
            continue
        stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
        effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
        registros.append({
            'janela':      f"{DIAS_JANELA}d",
            'feature':     nome_stat,
            'effect_size': round(effect, 4),
            'p_value':     round(p, 4),
        })

df_effect = pd.DataFrame(registros)

# Heatmap de effect size
pivot = df_effect.pivot(index='feature', columns='janela', values='effect_size')
pivot = pivot[[f"{d}d" for d in JANELAS]]  # ordena colunas

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    text=np.round(pivot.values, 3),
    texttemplate="%{text}",
    colorbar=dict(title="Effect Size")
))
fig.update_layout(
    title="Effect Size (Mann-Whitney) por Feature e Janela<br>"
          "<sup>Verde = alta separabilidade entre Real=0 e Real=1</sup>",
    template="plotly_white",
    xaxis_title="Janela", yaxis_title="Feature"
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}
n_metodos = len(metodos)

fig = make_subplots(
    rows=n_metodos, cols=len(JANELAS),
    subplot_titles=[
        f"{m['titulo']} — {d} dias"
        for m in metodos
        for d in JANELAS
    ],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
        dados_por_classe = {0: [], 1: []}

        for _, evento in df_eventos_crystallizer2.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
            valores = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(valores) < 2:
                continue
            classe = int(evento["Real"])
            dados_por_classe[classe].extend(valores.tolist())

        for classe, valores in dados_por_classe.items():
            fig.add_trace(go.Box(
                y=valores,
                name=nomes_classe[classe],
                marker_color=cores_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and col_idx == 1),
                boxmean='sd'
            ), row=row_idx, col=col_idx)

        fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
        fig.update_xaxes(title_text=metodo["titulo"], row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    boxmode="group",
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela — Crystallizer #2"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer2_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 (Eventos após 01/04/2020)"
)
fig.show()

## Crystallizer #3

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",         "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10",   "df": df_crystallizer3_0a10},
    {"titulo": "IQR",              "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",           "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

resultados = []
for DIAS_JANELA in JANELAS:
    vals = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer3.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) >= 2:
            vals[int(evento["Real"])].extend(v.tolist())

    v0, v1 = np.array(vals[0]), np.array(vals[1])

    # Mann-Whitney: diferença de medianas
    stat_mw, p_mw = mannwhitneyu(v0, v1, alternative='two-sided')
    # Kolmogorov-Smirnov: diferença na forma inteira da distribuição
    stat_ks, p_ks = ks_2samp(v0, v1)
    # Effect size (rank-biserial)
    effect = 1 - (2 * stat_mw) / (len(v0) * len(v1))

    resultados.append({
        'janela':      f"{DIAS_JANELA}D",
        'p_mw':        round(p_mw, 4),
        'p_ks':        round(p_ks, 4),
        'effect_size': round(abs(effect), 4),
        'media_r0':    round(np.mean(v0), 3),
        'media_r1':    round(np.mean(v1), 3),
        'delta_media': round(np.mean(v1) - np.mean(v0), 3),
    })

df_testes = pd.DataFrame(resultados)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

STATS_FUNCS = {
    'media':    np.mean,
    'mediana':  np.median,
    'std':      np.std,
    'max':      np.max,
    'p75':      lambda x: np.percentile(x, 75),
    'p90':      lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range':    lambda x: np.max(x) - np.min(x),
}

registros = []

for DIAS_JANELA in JANELAS:
    stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}

    for _, evento in df_eventos_crystallizer3.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df['TIMESTAMP'] >= inicio) &
            (df['TIMESTAMP'] <  ts)
        )
        v = df[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(v) < 2:
            continue
        classe = int(evento["Real"])
        for nome_stat, func in STATS_FUNCS.items():
            stat_vals[nome_stat][classe].append(func(v))

    for nome_stat in STATS_FUNCS:
        v0 = np.array(stat_vals[nome_stat][0])
        v1 = np.array(stat_vals[nome_stat][1])
        if len(v0) < 2 or len(v1) < 2:
            continue
        stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
        effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
        registros.append({
            'janela':      f"{DIAS_JANELA}d",
            'feature':     nome_stat,
            'effect_size': round(effect, 4),
            'p_value':     round(p, 4),
        })

df_effect = pd.DataFrame(registros)

# Heatmap de effect size
pivot = df_effect.pivot(index='feature', columns='janela', values='effect_size')
pivot = pivot[[f"{d}d" for d in JANELAS]]  # ordena colunas

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    text=np.round(pivot.values, 3),
    texttemplate="%{text}",
    colorbar=dict(title="Effect Size")
))
fig.update_layout(
    title="Effect Size (Mann-Whitney) por Feature e Janela<br>"
          "<sup>Verde = alta separabilidade entre Real=0 e Real=1</sup>",
    template="plotly_white",
    xaxis_title="Janela", yaxis_title="Feature"
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}
n_metodos = len(metodos)

fig = make_subplots(
    rows=n_metodos, cols=len(JANELAS),
    subplot_titles=[
        f"{m['titulo']} — {d} dias"
        for m in metodos
        for d in JANELAS
    ],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
        dados_por_classe = {0: [], 1: []}

        for _, evento in df_eventos_crystallizer3.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
            valores = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(valores) < 2:
                continue
            classe = int(evento["Real"])
            dados_por_classe[classe].extend(valores.tolist())

        for classe, valores in dados_por_classe.items():
            fig.add_trace(go.Box(
                y=valores,
                name=nomes_classe[classe],
                marker_color=cores_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and col_idx == 1),
                boxmean='sd'
            ), row=row_idx, col=col_idx)

        fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
        fig.update_xaxes(title_text=metodo["titulo"], row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    boxmode="group",
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela — Crystallizer #3"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3_filtrado_until2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 (Eventos até 01/04/2020)"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Real vs Falso Positivo — {m['titulo']}" for m in metodos],
    vertical_spacing=0.06
)

for row_idx, metodo in enumerate(metodos, start=1):
    df_atual = metodo["df"]

    for DIAS_JANELA in JANELAS:
        for classe in [0, 1]:
            valores_classe = []

            for _, evento in df_eventos_crystallizer3_filtrado_after2020.iterrows():
                if int(evento["Real"]) != classe:
                    continue
                ts     = evento["TIMESTAMP"]
                inicio = ts - pd.Timedelta(days=DIAS_JANELA)
                mask   = (df_atual['TIMESTAMP'] >= inicio) & (df_atual['TIMESTAMP'] < ts)
                v = df_atual[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    valores_classe.extend(v.tolist())

            fig.add_trace(go.Violin(
                y=valores_classe,
                x=[f"{DIAS_JANELA}d"] * len(valores_classe),
                name=nomes_classe[classe],
                legendgroup=nomes_classe[classe],
                showlegend=(row_idx == 1 and DIAS_JANELA == JANELAS[0]),
                side="negative" if classe == 0 else "positive",
                line_color=cores_classe[classe],
                meanline_visible=True,
                points=False
            ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Fe (ppm)", row=row_idx, col=1)
    fig.update_xaxes(title_text="Janela", row=row_idx, col=1)

fig.update_layout(
    height=500 * n_metodos,
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 (Eventos após 01/04/2020)"
)
fig.show()

# Comparação dos 3 reatores

## Verificando effect size por reator

In [ ]:
# Comparação de effect size entre crystallizers
crystallizers_config = [
    (df_crystallizer1, df_eventos_crystallizer1, "Crystallizer #1"),
    (df_crystallizer2, df_eventos_crystallizer2, "Crystallizer #2"),
    (df_crystallizer3, df_eventos_crystallizer3, "Crystallizer #3"),
]

STATS_FUNCS = {
    'media': np.mean, 'mediana': np.median, 'std': np.std,
    'max': np.max, 'p75': lambda x: np.percentile(x, 75),
    'p90': lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range': lambda x: np.max(x) - np.min(x),
}

registros_todos = []

for df_c, df_ev, nome_c in crystallizers_config:
    for DIAS_JANELA in JANELAS:
        stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}
        for _, evento in df_ev.iterrows():
            ts = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask = (df_c['TIMESTAMP'] >= inicio) & (df_c['TIMESTAMP'] < ts)
            v = df_c[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) < 2:
                continue
            classe = int(evento["Real"])
            for nome_stat, func in STATS_FUNCS.items():
                stat_vals[nome_stat][classe].append(func(v))

        for nome_stat in STATS_FUNCS:
            v0 = np.array(stat_vals[nome_stat][0])
            v1 = np.array(stat_vals[nome_stat][1])
            if len(v0) < 2 or len(v1) < 2:
                continue
            stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
            effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
            registros_todos.append({
                'crystallizer': nome_c,
                'janela': f"{DIAS_JANELA}d",
                'feature': nome_stat,
                'effect_size': round(effect, 4),
                'p_value': round(p, 4),
            })

df_effect_todos = pd.DataFrame(registros_todos)

# Heatmap com facetas por crystallizer
fig = make_subplots(rows=1, cols=3,
    subplot_titles=[c[2] for c in crystallizers_config])

for col_idx, (_, _, nome_c) in enumerate(crystallizers_config, start=1):
    subset = df_effect_todos[df_effect_todos['crystallizer'] == nome_c]
    pivot  = subset.pivot(index='feature', columns='janela', values='effect_size')
    pivot  = pivot[[f"{d}d" for d in JANELAS]]
    fig.add_trace(go.Heatmap(
        z=pivot.values, x=pivot.columns.tolist(), y=pivot.index.tolist(),
        colorscale='RdYlGn', zmin=0, zmax=1,
        text=np.round(pivot.values, 3), texttemplate="%{text}",
        showscale=(col_idx == 3)
    ), row=1, col=col_idx)

fig.update_layout(
    title="Effect Size comparativo — #1, #2 e #3",
    template="plotly_white", height=450
)
fig.show()

In [ ]:
crystallizers_config = [
    (df_crystallizer1,        df_eventos_crystallizer1, "C1 Original"),
    (df_crystallizer1_0a10,   df_eventos_crystallizer1, "C1 0-10"),
    (df_crystallizer1_iqr,    df_eventos_crystallizer1, "C1 IQR"),
    (df_crystallizer1_hampel, df_eventos_crystallizer1, "C1 Hampel"),
    (df_crystallizer2,        df_eventos_crystallizer2, "C2 Original"),
    (df_crystallizer2_0a10,   df_eventos_crystallizer2, "C2 0-10"),
    (df_crystallizer2_iqr,    df_eventos_crystallizer2, "C2 IQR"),
    (df_crystallizer2_hampel, df_eventos_crystallizer2, "C2 Hampel"),
    (df_crystallizer3,        df_eventos_crystallizer3, "C3 Original"),
    (df_crystallizer3_0a10,   df_eventos_crystallizer3, "C3 0-10"),
    (df_crystallizer3_iqr,    df_eventos_crystallizer3, "C3 IQR"),
    (df_crystallizer3_hampel, df_eventos_crystallizer3, "C3 Hampel"),
]

STATS_FUNCS = {
    'media'   : np.mean,
    'mediana' : np.median,
    'std'     : np.std,
    'max'     : np.max,
    'p75'     : lambda x: np.percentile(x, 75),
    'p90'     : lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range'   : lambda x: np.max(x) - np.min(x),
}

registros_todos = []
for df_c, df_ev, nome_c in crystallizers_config:
    for DIAS_JANELA in JANELAS:
        stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}
        for _, evento in df_ev.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_c['TIMESTAMP'] >= inicio) & (df_c['TIMESTAMP'] < ts)
            v      = df_c[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) < 2:
                continue
            classe = int(evento["Real"])
            for nome_stat, func in STATS_FUNCS.items():
                stat_vals[nome_stat][classe].append(func(v))

        for nome_stat in STATS_FUNCS:
            v0 = np.array(stat_vals[nome_stat][0])
            v1 = np.array(stat_vals[nome_stat][1])
            if len(v0) < 2 or len(v1) < 2:
                continue
            stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
            effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
            registros_todos.append({
                'crystallizer': nome_c,
                'janela'      : f"{DIAS_JANELA}d",
                'feature'     : nome_stat,
                'effect_size' : round(effect, 4),
                'p_value'     : round(p, 4),
            })

df_effect_todos = pd.DataFrame(registros_todos)

# Heatmap — 3 crystallizers × 4 métodos = 12 subplots
n_cols = 4  # Original, 0-10, IQR, Hampel
n_rows = 3  # C1, C2, C3
nomes  = [c[2] for c in crystallizers_config]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=nomes,
    vertical_spacing=0.08
)

for idx, (_, _, nome_c) in enumerate(crystallizers_config):
    row_idx = idx // n_cols + 1
    col_idx = idx % n_cols + 1

    subset = df_effect_todos[df_effect_todos['crystallizer'] == nome_c]
    pivot  = subset.pivot(index='feature', columns='janela', values='effect_size')
    pivot  = pivot[[f"{d}d" for d in JANELAS]]

    fig.add_trace(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        text=np.round(pivot.values, 3),
        texttemplate="%{text}",
        showscale=(col_idx == n_cols and row_idx == n_rows)
    ), row=row_idx, col=col_idx)

fig.update_layout(
    title="Effect Size comparativo — C1, C2, C3 × Original, 0-10, IQR, Hampel",
    template="plotly_white",
    height=400 * n_rows
)
fig.show()

## Análise antes/depois de 2020
Teste se as distribuições antes/depois de 2020 são diferentes dentro da mesma classe (Real=1 antes vs Real=1 depois)

In [ ]:
for crystallizer_nome, df_c, df_ev_full in [
    ("Crystallizer #1", df_crystallizer1, df_eventos_crystallizer1),
    ("Crystallizer #2", df_crystallizer2, df_eventos_crystallizer2),
    ("Crystallizer #3", df_crystallizer3, df_eventos_crystallizer3),
]:
    df_antes  = df_ev_full[df_ev_full["TIMESTAMP"] <= pd.to_datetime("2020-04-01")]
    df_depois = df_ev_full[df_ev_full["TIMESTAMP"] >  pd.to_datetime("2020-04-01")]

    print(f"\n{'='*50}")
    print(f"{crystallizer_nome}")
    print(f"  Eventos antes 2020:  {len(df_antes)}  (Real=1: {(df_antes['Real']==1).sum()})")
    print(f"  Eventos depois 2020: {len(df_depois)} (Real=1: {(df_depois['Real']==1).sum()})")

    DIAS_JANELA = 7
    for classe in [0, 1]:
        vals_antes, vals_depois = [], []
        for df_periodo, lista in [(df_antes, vals_antes), (df_depois, vals_depois)]:
            for _, ev in df_periodo[df_periodo["Real"] == classe].iterrows():
                ts = ev["TIMESTAMP"]
                mask = (df_c['TIMESTAMP'] >= ts - pd.Timedelta(days=DIAS_JANELA)) & \
                       (df_c['TIMESTAMP'] < ts)
                v = df_c[mask]["Resultado de Ferro (ppm)"].dropna().values
                if len(v) >= 2:
                    lista.extend(v.tolist())

        if len(vals_antes) >= 5 and len(vals_depois) >= 5:
            stat, p = ks_2samp(vals_antes, vals_depois)
            print(f"  Classe={classe} — KS p-value antes vs depois 2020: {p:.4f} "
                  f"({'DIFERENTE' if p < 0.05 else 'similar'})")

## Análise de Intervalo Entre Eventos (gap temporal)

In [ ]:
for nome_c, df_ev in [
    ("Crystallizer #1", df_eventos_crystallizer1),
    ("Crystallizer #2", df_eventos_crystallizer2),
    ("Crystallizer #3", df_eventos_crystallizer3),
]:
    df_real1 = df_ev[df_ev["Real"] == 1].sort_values("TIMESTAMP")
    gaps = df_real1["TIMESTAMP"].diff().dt.days.dropna()

    print(f"\n{nome_c} — intervalos entre eventos reais (dias):")
    print(f"  Mediana: {gaps.median():.0f}  |  Min: {gaps.min():.0f}  |  Max: {gaps.max():.0f}")
    print(f"  Tendência (slope dos gaps): {np.polyfit(range(len(gaps)), gaps, 1)[0]:.2f} dias/evento")
    # Slope negativo = intervalos diminuindo = aceleração de falhas

## Correlação cruzada entre os crystallizers

In [ ]:
def remove_outliers_iqr(df, colunas, fator=1.5):
    mask = pd.Series(True, index=df.index)
    for col in colunas:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        mask &= df[col].between(Q1 - fator * IQR, Q3 + fator * IQR)
    return df[mask]

def add_scatter_regressao(fig, x, y, row, col):
    lr = LinearRegression().fit(x.reshape(-1, 1), y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = lr.predict(x_line.reshape(-1, 1))
    r2 = lr.score(x.reshape(-1, 1), y)

    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(size=6, opacity=0.6, color='steelblue'),
        showlegend=False
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode='lines',
        line=dict(color='red', width=2),
        showlegend=False
    ), row=row, col=col)

    fig.add_annotation(
        x=x.min(), y=y.max(),
        text=f"R² = {r2:.3f}",
        showarrow=False,
        font=dict(size=12, color='red'),
        xanchor='left',
        row=row, col=col
    )

# ── Prepara dados ─────────────────────────────────────────────────────────────
df_c1_idx = df_crystallizer1.set_index("TIMESTAMP")["Resultado de Ferro (ppm)"]
df_c2_idx = df_crystallizer2.set_index("TIMESTAMP")["Resultado de Ferro (ppm)"]
df_c3_idx = df_crystallizer3.set_index("TIMESTAMP")["Resultado de Ferro (ppm)"]

freq = "3D"
s1 = df_c1_idx.resample(freq).mean()
s2 = df_c2_idx.resample(freq).mean()
s3 = df_c3_idx.resample(freq).mean()
df_corr = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()
df_corr_clean = remove_outliers_iqr(df_corr, ["C1", "C2", "C3"], fator=1.5)

print("Correlação de Pearson — com outliers:")
print(df_corr.corr(method="pearson").round(3))
print(f"\nCorrelação de Pearson — sem outliers ({len(df_corr) - len(df_corr_clean)} removidos):")
print(df_corr_clean.corr(method="pearson").round(3))

# ── Figura com 2 linhas ───────────────────────────────────────────────────────
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        *[f"{a} vs {b}" for a, b in pares],
        *[f"{a} vs {b} (sem outliers)" for a, b in pares]
    ]
)

for col_idx, (a, b) in enumerate(pares, start=1):
    # Linha 1 — com outliers
    add_scatter_regressao(fig, df_corr[a].values, df_corr[b].values, row=1, col=col_idx)
    fig.update_xaxes(title_text=a, row=1, col=col_idx)
    fig.update_yaxes(title_text=b, row=1, col=col_idx)

    # Linha 2 — sem outliers
    add_scatter_regressao(fig, df_corr_clean[a].values, df_corr_clean[b].values, row=2, col=col_idx)
    fig.update_xaxes(title_text=a, row=2, col=col_idx)
    fig.update_yaxes(title_text=b, row=2, col=col_idx)

fig.update_layout(
    height=900,
    template='plotly_white',
    title="Correlação cruzada entre Crystallizers #1, #2, #3"
)
fig.show()

In [ ]:
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]
testes = [
    ("Pearson",   lambda x, y: pearsonr(x, y)),
    ("Spearman",  lambda x, y: spearmanr(x, y)),
    ("Kendall",   lambda x, y: kendalltau(x, y)),
]

registros = []

for label_df, df_teste in [("com outliers", df_corr), ("sem outliers", df_corr_clean)]:
    for a, b in pares:
        x = df_teste[a].values
        y = df_teste[b].values
        for nome_teste, func in testes:
            stat, p = func(x, y)
            registros.append({
                "dataset":    label_df,
                "par":        f"{a} vs {b}",
                "teste":      nome_teste,
                "coef":       round(stat, 4),
                "p_value":    round(p, 6),
                "sig_0.05":   "✓" if p < 0.05 else "✗",
                "sig_0.01":   "✓" if p < 0.01 else "✗",
                "n":          len(x),
            })

df_testes = pd.DataFrame(registros)

print("=" * 75)
print("TESTES DE CORRELAÇÃO — Crystallizers #1, #2, #3")
print("=" * 75)
for label_df in ["com outliers", "sem outliers"]:
    print(f"\n── {label_df.upper()} ──")
    sub = df_testes[df_testes["dataset"] == label_df]
    print(sub[["par","teste","coef","p_value","sig_0.05","sig_0.01","n"]]
          .to_string(index=False))

In [ ]:
# Cross-correlação com defasagem (lag) entre os crystallizers
MAX_LAG = 15  # dias — ajuste conforme DIAS_JANELA do seu modelo

lags      = range(-MAX_LAG, MAX_LAG + 1)
resultados_lag = []

for a, b in [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]:
    x = df_corr_clean[a].values
    y = df_corr_clean[b].values

    for lag in lags:
        if lag < 0:
            xs, ys = x[:lag],  y[-lag:]
        elif lag > 0:
            xs, ys = x[lag:],  y[:-lag]
        else:
            xs, ys = x, y

        r, p = pearsonr(xs, ys)
        resultados_lag.append({
            "par": f"{a} vs {b}", "lag_periodos": lag,
            "lag_dias": lag * 3,   # freq="3D"
            "r": round(r, 4), "p": round(p, 6)
        })

df_lag = pd.DataFrame(resultados_lag)

# Plot
fig = go.Figure()
for par in df_lag["par"].unique():
    sub = df_lag[df_lag["par"] == par]
    fig.add_trace(go.Scatter(
        x=sub["lag_dias"], y=sub["r"],
        mode="lines", name=par, line=dict(width=2)
    ))

fig.add_vline(x=0, line_dash="dash", line_color="black",
              annotation_text="lag=0")
fig.add_hline(y=0, line_color="gray", line_width=0.5)

fig.update_layout(
    title="Correlação cruzada com Defasagem entre Reatores<br>"
          "<sup>Pico em lag≠0 indica que um reator influencia o outro</sup>",
    xaxis_title="Defasagem (dias) — negativo: A influencia B  |  positivo: B influencia A",
    yaxis_title="Correlação de Pearson",
    template="plotly_white", height=450, hovermode="x unified",
    yaxis_range=[0.4,1]
)
fig.show()

# Lag de máxima correlação por par
print("\nLag de máxima correlação por par:")
for par in df_lag["par"].unique():
    sub  = df_lag[df_lag["par"] == par]
    best = sub.loc[sub["r"].idxmax()]
    print(f"  {par}: lag={best['lag_dias']:.0f} dias  |  r={best['r']:.4f}")

# Clusterização (Não supervisionada)

### Extração de Features

In [ ]:
def extract_features(df_window, df_baseline=None):
    """
    Extrai características da janela para detecção de anomalias.

    Parâmetros
    ----------
    df_window   : DataFrame com colunas TIMESTAMP e 'Resultado de Ferro (ppm)'
                  contendo os dados da janela anterior ao evento.
    df_baseline : DataFrame com o mesmo schema, contendo o período de
                  referência histórica (DIAS_BASELINE dias antes da janela).
                  Se None, as features de contexto histórico recebem NaN.
    """
    if len(df_window) < 5:
        return None

    #  Vetores base 
    t_hours = (df_window['TIMESTAMP'] - df_window['TIMESTAMP'].min()) \
                .dt.total_seconds() / 3600.0
    y_ppm   = df_window['Resultado de Ferro (ppm)'].values

    #  Derivadas — primeira (taxa ppm/hora) 
    dt    = np.diff(t_hours)
    dy    = np.diff(y_ppm)
    rates = np.divide(dy, dt, out=np.zeros_like(dy), where=dt != 0)

    #  Derivadas — segunda (aceleração ppm/hora²) 
    if len(rates) > 1:
        aceleracoes      = np.diff(rates) / (dt[1:] + 1e-9)
        aceleracao_media = float(np.mean(aceleracoes))
        aceleracao_final = float(aceleracoes[-1])
    else:
        aceleracao_media = aceleracao_final = 0.0

    #  Inclinação linear global 
    lr    = LinearRegression().fit(t_hours.values.reshape(-1, 1), y_ppm)
    slope = float(lr.coef_[0])

    #  Integral 
    area = float(trapezoid(y=y_ppm, x=t_hours))

    #  Último movimento antes do evento 
    last_dy   = float(dy[-1]) if len(dy) > 0 else 0.0
    last_dt   = float(dt[-1]) if len(dt) > 0 else 0.0
    ema_final = float(
        df_window['Resultado de Ferro (ppm)']
        .ewm(span=len(df_window), adjust=False).mean().iloc[-1]
    )

    #  Complexidade / inversões de tendência 
    inversoes_tendencia = int(np.sum(np.diff(np.sign(rates)) != 0)) \
                          if len(rates) > 1 else 0

    #  Energia das oscilações 
    # Soma dos quadrados das variações normalizadas — captura magnitude das mudanças
    energia_oscilacao = float(np.sum((dy / (np.std(y_ppm) + 1e-9)) ** 2))

    #  Picos 
    rms          = np.sqrt(np.mean(y_ppm ** 2))
    crest_factor = float(np.max(np.abs(y_ppm)) / rms) if rms > 0 else 0.0

    picos_idx, props = find_peaks(y_ppm, prominence=0)
    max_prominence   = float(np.max(props['prominences'])) if len(picos_idx) > 0 else 0.0

    # Picos com proeminência >= 1 desvio padrão (picos expressivos)
    picos_exp_idx, _ = find_peaks(y_ppm, prominence=float(np.std(y_ppm)))

    #  Comportamento pós-pico 
    # Captura se o sinal sustenta ou cai após o último pico expressivo
    if len(picos_exp_idx) > 0:
        idx_ultimo_pico         = picos_exp_idx[-1]
        y_apos                  = y_ppm[idx_ultimo_pico:]
        tempo_desde_ultimo_pico = float(t_hours.max() - t_hours.values[idx_ultimo_pico])
        media_apos_ultimo_pico  = float(np.mean(y_apos))
        # decay > 0: sinal caiu após o pico | decay < 0: sinal subiu (raro)
        decay_apos_pico         = float(y_apos[0] - y_apos[-1]) if len(y_apos) > 1 else 0.0
    else:
        tempo_desde_ultimo_pico = float(t_hours.max())
        media_apos_ultimo_pico  = float(np.mean(y_ppm))
        decay_apos_pico         = 0.0

    #  Razão pico / vale 
    vales_idx, _ = find_peaks(-y_ppm, prominence=0)
    media_picos  = float(np.mean(y_ppm[picos_idx])) if len(picos_idx) > 0 \
                   else float(np.max(y_ppm))
    media_vales  = float(np.mean(y_ppm[vales_idx])) if len(vales_idx) > 0 \
                   else float(np.min(y_ppm))
    ratio_pico_vale = media_picos / (media_vales + 1e-9)

    #  Shape da distribuição 
    ppm_skewness = float(skew(y_ppm))
    ppm_kurtosis = float(kurtosis(y_ppm))
    range_norm   = float((np.max(y_ppm) - np.min(y_ppm)) / (np.mean(y_ppm) + 1e-9))

    #  Tendência por metades 
    mid                   = len(y_ppm) // 2
    media_primeira_metade = float(np.mean(y_ppm[:mid])) if mid >= 1 else float(np.mean(y_ppm))
    media_segunda_metade  = float(np.mean(y_ppm[mid:])) if mid >= 1 else float(np.mean(y_ppm))
    ratio_metades         = media_segunda_metade / (media_primeira_metade + 1e-9)

    t_second = t_hours.values[mid:]
    y_second = y_ppm[mid:]
    if len(t_second) >= 2:
        slope_recente = float(
            LinearRegression().fit(t_second.reshape(-1, 1), y_second).coef_[0]
        )
    else:
        slope_recente = slope

    #  Tendência no terço final 
    # Mais granular que slope_recente — captura os últimos ~2 dias da janela de 7d
    terco   = len(y_ppm) * 2 // 3
    t_final = t_hours.values[terco:]
    y_final = y_ppm[terco:]
    if len(t_final) >= 2:
        slope_final = float(
            LinearRegression().fit(t_final.reshape(-1, 1), y_final).coef_[0]
        )
        nivel_final = float(np.mean(y_final))
    else:
        slope_final = slope_recente
        nivel_final = float(y_ppm[-1])

    #  Estacionariedade / mudança de patamar 
    # Mede se o sinal migrou para um nível diferente ao longo da janela.
    # > 0: subiu de patamar | < 0: desceu | ~ 0: estável
    terco_n     = max(len(y_ppm) // 3, 1)
    shift_nivel = float(
        (np.mean(y_ppm[2 * terco_n:]) - np.mean(y_ppm[:terco_n]))
        / (np.std(y_ppm) + 1e-9)
    )

    #  Tempo acima do limite operacional fixo 
    pct_acima_limite   = float(np.mean(y_ppm > threshold))
    # Integra o tempo (horas) em que o sinal ficou acima do limite
    acima_flag         = (y_ppm[:-1] > threshold).astype(float)
    horas_acima_limite = float(np.sum(acima_flag * dt))

    #  Regularidade temporal da amostragem 
    n_amostras    = len(y_ppm)
    dt_medio      = float(np.mean(dt))  if len(dt) > 0 else 0.0
    dt_min        = float(np.min(dt))   if len(dt) > 0 else 0.0
    dt_variancia  = float(np.var(dt))   if len(dt) > 1 else 0.0
    duracao_total = float(t_hours.max()) if t_hours.max() > 0 else 1e-9
    freq_amostras = n_amostras / (duracao_total + 1e-9)

    #  Contexto histórico (requer df_baseline) 
    if df_baseline is not None and len(df_baseline) >= 2:
        y_base         = df_baseline['Resultado de Ferro (ppm)'].values
        baseline_media = float(np.mean(y_base))
        baseline_std   = float(np.std(y_base))

        zscore_max          = (np.max(y_ppm)  - baseline_media) / (baseline_std + 1e-9)
        zscore_media        = (np.mean(y_ppm) - baseline_media) / (baseline_std + 1e-9)
        ratio_media         = np.mean(y_ppm) / (baseline_media + 1e-9)
        ratio_max           = np.max(y_ppm)  / (baseline_media + 1e-9)

        threshold_hist      = float(np.percentile(y_base, 75))
        n_acima_threshold   = int(np.sum(y_ppm > threshold_hist))
        pct_acima_threshold = float(np.mean(y_ppm > threshold_hist))

        flags  = np.concatenate([[False], y_ppm > threshold_hist, [False]])
        diffs  = np.diff(flags.astype(int))
        starts = np.where(diffs ==  1)[0]
        ends   = np.where(diffs == -1)[0]
        max_run_acima = int(np.max(ends - starts)) if len(starts) > 0 else 0

    else:
        zscore_max = zscore_media = ratio_media = ratio_max = np.nan
        n_acima_threshold = pct_acima_threshold = max_run_acima = np.nan

    #  Dicionário final 
    return {
        # Taxas de variação (1ª derivada)
        'taxa_max':                 float(np.max(rates)) if len(rates) > 0 else 0.0,
        'taxa_media':               float(np.mean(rates)) if len(rates) > 0 else 0.0,
        # Aceleração (2ª derivada)
        'aceleracao_media':         aceleracao_media,
        'aceleracao_final':         aceleracao_final,
        # Tendência global
        'slope':                    slope,
        'slope_recente':            slope_recente,
        'slope_final':              slope_final,
        'nivel_final':              nivel_final,
        # Estacionariedade
        'shift_nivel':              shift_nivel,
        # Magnitude e acumulação
        'area_curva':               area,
        'ppm_max':                  float(np.max(y_ppm)),
        'ppm_min':                  float(np.min(y_ppm)),
        'ppm_media':                float(np.mean(y_ppm)),
        'ppm_std':                  float(np.std(y_ppm)),
        # Último movimento
        'ultimo_dy':                last_dy,
        'ultimo_dt':                last_dt,
        'ema_final':                ema_final,
        # Complexidade e energia
        'inversoes_tendencia':      inversoes_tendencia,
        'energia_oscilacao':        energia_oscilacao,
        # Picos
        'crest_factor':             crest_factor,
        'max_prominence':           max_prominence,
        'tempo_desde_ultimo_pico':  tempo_desde_ultimo_pico,
        'media_apos_ultimo_pico':   media_apos_ultimo_pico,
        'decay_apos_pico':          decay_apos_pico,
        'ratio_pico_vale':          ratio_pico_vale,
        # Shape da distribuição
        'skewness':                 ppm_skewness,
        'kurtosis':                 ppm_kurtosis,
        'range_norm':               range_norm,
        # Tendência por metades
        'media_primeira_metade':    media_primeira_metade,
        'media_segunda_metade':     media_segunda_metade,
        'ratio_metades':            ratio_metades,
        # Limite operacional fixo
        'pct_acima_limite':         pct_acima_limite,
        'horas_acima_limite':       horas_acima_limite,
        # Amostragem
        'n_amostras':               n_amostras,
        'dt_medio':                 dt_medio,
        'dt_min':                   dt_min,
        'dt_variancia':             dt_variancia,
        'freq_amostras':            freq_amostras,
        # Contexto histórico
        'zscore_max':               zscore_max,
        'zscore_media':             zscore_media,
        'ratio_media':              ratio_media,
        'ratio_max':                ratio_max,
        'n_acima_threshold':        n_acima_threshold,
        'pct_acima_threshold':      pct_acima_threshold,
        'max_run_acima':            max_run_acima,
    }


# =============================================================================
# EXTRAÇÃO DE FEATURES POR JANELA
# =============================================================================

features_list      = []
valid_events       = []
eventos_timestamps = df_eventos["TIMESTAMP"]

for evento in eventos_timestamps:
    inicio_janela   = evento - pd.Timedelta(days=DIAS_JANELA)
    inicio_baseline = evento - pd.Timedelta(days=DIAS_JANELA + DIAS_BASELINE)

    # Janela principal
    mask_window = (
        (df_dataset['TIMESTAMP'] >= inicio_janela) &
        (df_dataset['TIMESTAMP'] <  evento)
    )
    df_window = df_dataset[mask_window]

    # Janela de baseline histórico
    mask_baseline = (
        (df_dataset['TIMESTAMP'] >= inicio_baseline) &
        (df_dataset['TIMESTAMP'] <  inicio_janela)
    )
    df_base = df_dataset[mask_baseline]

    baseline_arg = df_base if len(df_base) >= 2 else None

    feats = extract_features(df_window, df_baseline=baseline_arg)
    if feats:
        features_list.append(feats)
        valid_events.append(evento)

# =============================================================================
# CRIAÇÃO DO DATAFRAME DE FEATURES
# =============================================================================

df_features = pd.DataFrame(features_list, index=valid_events)

# Adiciona coluna 'Real' do df_eventos
df_real = df_eventos.set_index("TIMESTAMP")["Real"]
df_features["Real"] = df_real.reindex(df_features.index)

# Preenche NaN das features de baseline com a mediana da coluna
cols_baseline = [
    'zscore_max', 'zscore_media', 'ratio_media', 'ratio_max',
    'n_acima_threshold', 'pct_acima_threshold', 'max_run_acima'
]
for col in cols_baseline:
    if df_features[col].isna().any():
        df_features[col] = df_features[col].fillna(df_features[col].median())

# # =============================================================================
# # ISOLATION FOREST SCORE COMO FEATURE EXTRA
# # =============================================================================

# feature_cols = [col for col in df_features.columns if col != "Real"]
# X = df_features[feature_cols].values
# y = df_features["Real"].values

# X_negativos = X[y == 0]
# iso = IsolationForest(contamination=0.05, random_state=42)
# iso.fit(X_negativos)

# df_features['iso_score'] = iso.decision_function(X)

df_features

### Escalonamento dos dados

In [ ]:
# Padronização e Clusterização
scaler = StandardScaler() #StandardScaler || RobustScaler || MinMaxScaler
X_scaled = scaler.fit_transform(df_features.drop('Real', axis=1))
y = df_features['Real']

### Verificando separabilidade das amostras com PCA

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

cores = {0: 'steelblue', 1: 'crimson'}
labels_texto = {0: 'Sem Evento apontado', 1: 'Evento apontado'}

fig, ax = plt.subplots(figsize=(8, 6))
for val in [0, 1]:
    mask = df_features['Real'] == val
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=cores[val], label=labels_texto[val],
               s=100, edgecolors='k', linewidths=0.8)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)')
ax.legend()
ax.set_title('Separabilidade das classes')
plt.tight_layout()
plt.show()

## KMeans

In [ ]:
# Elbow Method
inercia = []
K_range = range(1, 8)

for k in K_range:
    kmeans_teste = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_teste.fit(X_scaled)
    inercia.append(kmeans_teste.inertia_)

# Plota o gráfico para visualização
plt.figure(figsize=(10,5))
plt.plot(K_range, inercia, marker='o', linestyle='--')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo para K Ideal')
plt.xticks(K_range)
plt.grid(True)
plt.show()

In [ ]:
# Clusterização
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto') # Como são duas classes, o número de clusters foi fixado como 2
df_features['Cluster'] = kmeans.fit_predict(X_scaled)
# Resultado final
df_features

In [ ]:
# Dicionário de cores dos clusters
cores_clusters = {
    0: 'blue',   
    1: 'green',
    2: 'red'
}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['TIMESTAMP'],
    y=df_dataset["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos.iterrows():
    evento_ts = row["TIMESTAMP"]
    
    # Adiciona o sombreado da janela baseado no Cluster
    if evento_ts in df_features.index:
        cluster = df_features.loc[evento_ts, 'Cluster']
        cor_fundo = cores_clusters.get(cluster, 'gray')
        inicio_janela = evento_ts - pd.Timedelta(days=DIAS_JANELA)
        
        fig.add_vrect(
            x0=inicio_janela,
            x1=evento_ts,
            fillcolor=cor_fundo,
            opacity=0.2, 
            layer="below", 
            line_width=0,
            annotation_text=f"C{cluster}",
            annotation_position="top left"
        )

    # Linha vertical exata do evento
    cor = "red" if row["Real"] == 1 else "blue"
    fig.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )

    # Anotação
    fig.add_annotation(
        x=str(evento_ts),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title='Teor de Fe - Clusters'
)
fig.show()

In [ ]:
# Tabela de contingência
pd.crosstab(df_features['Cluster'], df_features['Real'], 
            rownames=['Cluster'], colnames=['Real'])

In [ ]:
## Quão bem os clusters recuperam os labels reais
ari  = adjusted_rand_score(df_features['Real'], df_features['Cluster'])
nmi  = normalized_mutual_info_score(df_features['Real'], df_features['Cluster'])

print(f"Adjusted Rand Score: {ari}")
print(f"Normalized Mutual Info Score: {ari}")

# Classificação (Abordagem Supervisionada)